# 04 — Visualisasi Embedding IndoSBERT (Bukti Transformer Memahami Makna)

Notebook ini **mendemonstrasikan secara visual** apa yang dilakukan Transformer (IndoSBERT):
mengubah teks pasal menjadi vektor 768-dimensi yang **mengelompok berdasarkan makna**.

Jika pasal-pasal dari domain yang sama saling berdekatan di ruang embedding, itu bukti
bahwa Transformer menangkap *semantik*, bukan sekadar kata.

**Prasyarat:** sudah menjalankan `scripts/03_build_index.py` (index FAISS tersedia).

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import faiss
import matplotlib.pyplot as plt
from src.config import load_config, resolve_path

cfg = load_config()
index_dir = resolve_path(cfg['paths']['index_dir'])

# Ambil kembali seluruh embedding dari index FAISS (hasil encode IndoSBERT)
index = faiss.read_index(str(index_dir / 'faiss.faiss'))
emb = index.reconstruct_n(0, index.ntotal)
meta = json.loads((index_dir / 'faiss.meta.json').read_text(encoding='utf-8'))
ids = meta['doc_ids']
domains = [i.split('_')[0] for i in ids]
print('Jumlah embedding:', emb.shape[0], '| dimensi:', emb.shape[1])

## 1. Reduksi Dimensi dengan PCA (768-dim → 2-dim)
PCA cepat dan mempertahankan struktur global.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42).fit_transform(emb)

palette = {'KETENAGAKERJAAN': '#4C72B0', 'KONSUMEN': '#55A868',
           'ITE': '#C44E52', 'ANAK': '#8172B3'}

def scatter(coords, title):
    plt.figure(figsize=(9, 7))
    for dom, color in palette.items():
        m = [d == dom for d in domains]
        plt.scatter(coords[m, 0], coords[m, 1], s=14, alpha=0.6,
                    c=color, label=dom.capitalize())
    plt.title(title); plt.legend(title='Domain Hukum'); plt.tight_layout(); plt.show()

scatter(pca, 'Embedding IndoSBERT (PCA) — diwarnai per domain')

## 2. Reduksi Dimensi dengan t-SNE (struktur lokal lebih jelas)
t-SNE lebih baik menampilkan **klaster** — perhatikan pasal sejenis berkumpul.
(Perlu waktu ~10–30 detik.)

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, init='pca',
            learning_rate='auto', random_state=42).fit_transform(emb)
scatter(tsne, 'Embedding IndoSBERT (t-SNE) — pasal mengelompok berdasarkan makna')

**Interpretasi:** bila titik-titik dengan warna sama saling berkumpul, artinya IndoSBERT
menempatkan pasal-pasal bertema serupa berdekatan di ruang vektor — *tanpa* pernah diberi tahu
label domainnya. Inilah bukti Transformer memahami makna.

## 3. Demonstrasi: Query Bahasa Awam Mendarat di Klaster yang Tepat
Kita encode sebuah query sehari-hari, lalu lihat ia jatuh di dekat domain mana.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(cfg['embedding']['model_name'],
                            cache_folder=str(resolve_path(cfg['paths']['models_dir'])))

queries = ['berapa pesangon kalau dipecat?',
           'ditipu saat belanja online',
           'anak menjadi korban kekerasan']
q_emb = model.encode(queries, normalize_embeddings=True)

# Proyeksikan query ke ruang PCA yang sama
pca_model = PCA(n_components=2, random_state=42).fit(emb)
q_pca = pca_model.transform(q_emb)
doc_pca = pca_model.transform(emb)

plt.figure(figsize=(9, 7))
for dom, color in palette.items():
    m = [d == dom for d in domains]
    plt.scatter(doc_pca[m, 0], doc_pca[m, 1], s=12, alpha=0.35, c=color, label=dom.capitalize())
plt.scatter(q_pca[:, 0], q_pca[:, 1], s=220, c='black', marker='*', zorder=5, label='QUERY')
for (x, y), q in zip(q_pca, queries):
    plt.annotate(q, (x, y), fontsize=8, xytext=(6, 6), textcoords='offset points')
plt.title('Query (bintang) mendarat dekat domain yang relevan')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Heatmap Kemiripan Semantik (cosine similarity)
Membandingkan beberapa kalimat: yang **bermakna mirip** punya skor tinggi walau kata berbeda.

In [ ]:
kalimat = [
    'pekerja dipecat dari perusahaan',
    'pemutusan hubungan kerja terhadap buruh',
    'konsumen membeli barang yang rusak',
    'pembeli menerima produk cacat',
]
E = model.encode(kalimat, normalize_embeddings=True)
sim = E @ E.T  # cosine (sudah ternormalisasi)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(kalimat))); ax.set_yticks(range(len(kalimat)))
ax.set_xticklabels([k[:22] for k in kalimat], rotation=40, ha='right', fontsize=8)
ax.set_yticklabels([k[:22] for k in kalimat], fontsize=8)
for i in range(len(kalimat)):
    for j in range(len(kalimat)):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                color='white' if sim[i,j] < 0.7 else 'black', fontsize=9)
plt.colorbar(im, label='cosine similarity')
plt.title('Kemiripan Semantik antar-kalimat (IndoSBERT)')
plt.tight_layout(); plt.show()

**Perhatikan:** kalimat 1↔2 ("dipecat" vs "pemutusan hubungan kerja") dan 3↔4
("barang rusak" vs "produk cacat") berskor tinggi meski **kata-katanya berbeda** —
tepat seperti yang dijelaskan di `docs/penjelasan_transformer.md`. Inilah kemampuan
yang tidak dimiliki BM25, dan alasan kita memerlukan Transformer.